# P3 · How we observe

Astronomy has exactly one input: **light that arrives**. (Plus, recently, neutrinos and
gravitational waves.) Everything else is inference. So it's worth understanding what
governs how much light you catch, how precisely you can measure it, and what fundamentally
limits you.

**You'll learn:** what each band of the spectrum reveals · why some telescopes must be in
space · the four things you can measure · what sets resolution · and where photometric noise
actually comes from.

## 1. The electromagnetic spectrum is a set of different questions

Each band is produced by different physics, so each answers a different question. This is
why big observatories come in sets.

| Band | Wavelength | What emits it | Example facility |
|---|---|---|---|
| **Radio** | > 1 mm | Cold gas, magnetic fields, pulsars | VLA, ALMA, CHIME |
| **Infrared** | 700 nm – 1 mm | Cool dust, redshifted early galaxies, planet formation | JWST, Spitzer |
| **Optical** | 380–750 nm | Stellar photospheres | Hubble, Kepler, TESS, Rubin |
| **Ultraviolet** | 10–380 nm | Hot young stars, accretion | Hubble, GALEX |
| **X-ray** | 0.01–10 nm | Million-degree gas, accretion onto compact objects | Chandra, XMM |
| **Gamma ray** | < 0.01 nm | The most violent processes known | Fermi, Cherenkov arrays |

**A cool object and a hot object are literally different subjects.** A dust cloud at 30 K
peaks in the far infrared and is invisible optically. Gas at 10⁷ K peaks in X-rays. To
observe a *system* you need multiple bands, which is why "multi-wavelength" is a standing
requirement rather than a bonus.

## 2. Why space telescopes exist

The atmosphere is opaque to most of the spectrum. That's excellent for life and inconvenient
for astronomy.

- **Blocked completely:** gamma ray, X-ray, most UV (ozone), most infrared (water vapour)
- **Gets through:** optical, some near-IR windows, radio

So X-ray and UV astronomy are only possible from space — not "better", *possible*. And even
in the optical, where light does get through, the atmosphere still hurts:

- **Seeing.** Turbulence smears a point source into a blob ~1 arcsecond across, throwing
  away most of a large telescope's theoretical resolution. Adaptive optics fights this.
- **Scintillation.** The same turbulence makes brightness flicker — which is fatal for
  precise photometry. This is *the* reason Kepler and TESS are in space: not to see fainter
  things, but to measure brightness *stably*. A 1% transit is undetectable through
  atmospheric flicker from the ground.

That connects directly to your work: the reason the data in this repo is so clean is that
nothing was between the star and the detector.

## 3. The four things you can measure

Nearly all observational astronomy is one of these, or a combination:

| Measurement | Question it answers | Data product |
|---|---|---|
| **Photometry** | How bright? | A number per exposure → **light curve** |
| **Spectroscopy** | Brightness at each wavelength? | **Spectrum** → composition, temperature, velocity |
| **Astrometry** | Where exactly, and moving how? | Positions over time → distance, proper motion |
| **Polarimetry** | Is the light polarised? | Magnetic fields, scattering geometry |

Photometry is the cheapest and highest-cadence, which is why time-domain surveys are
photometric. Spectroscopy carries far more information per target but costs enormously more
light — you're splitting the same photons across hundreds of wavelength bins.

That trade-off shapes the whole exoplanet field: **find** candidates photometrically because
it's cheap and parallel, then **confirm** them spectroscopically because that's where the
mass is, one target at a time.

## 4. Resolution: what "aperture" buys

Two separate things scale with telescope size:

**Light-collecting area** goes as diameter², so a 10 m mirror gathers 25× the photons of a
2 m. This buys faintness and precision.

**Angular resolution** is set by diffraction: θ ≈ 1.22 λ/D radians. Bigger aperture and
shorter wavelength both give finer detail. This is a hard physical limit, not an engineering
one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from skyplay import plotting

plotting.use_style()

def diffraction_limit(diameter_m, wavelength_nm):
    theta = 1.22 * (wavelength_nm * 1e-9) / diameter_m      # radians
    return (theta * u.rad).to(u.arcsec)

for name, d, wl in [('Human eye (7 mm pupil)', 0.007, 550),
                    ('Amateur 8-inch (0.2 m)', 0.20, 550),
                    ('Hubble (2.4 m)', 2.4, 550),
                    ('JWST (6.5 m, near-IR)', 6.5, 2000),
                    ('VLT (8.2 m)', 8.2, 550),
                    ('ELT (39 m)', 39.0, 550)]:
    print(f'  {name:26s} {diffraction_limit(d, wl).value:8.3f} arcsec')

print()
print('  Ground-based seeing typically ~1 arcsec  <- the atmosphere throws away')
print('  everything past the 0.2 m mark unless you use adaptive optics.')
print()
print('For scale: 1 arcsec is a coin seen from ~4 km. Resolving an Earth-like planet')
print('next to its star needs milli-arcseconds -- which is why transits (a brightness')
print('measurement) got there first. We never resolve the planet at all.')

## 5. Where the noise comes from

This is the most practically useful section, because it tells you what precision is
achievable before you try.

**Photon noise (shot noise) is the floor.** Light arrives as discrete photons at random
times. Counting N photons gives an uncertainty of √N, so the *fractional* precision is
√N/N = 1/√N. Want 10× better precision? Collect 100× more photons.

This is not an instrumental defect. It is the quantum nature of light, and no amount of
engineering removes it.

Other contributions, all of which you can in principle reduce:

- **Read noise** — added by the detector each time it's read out. Favours longer exposures.
- **Dark current** — thermally generated electrons. Why detectors are cooled.
- **Background** — sky brightness, scattered light, zodiacal light.
- **Systematics** — pointing drift, temperature changes, sensitivity variations. *Usually the
  real limit*, and the reason notebook 03 exists.

In [ ]:
rng = np.random.default_rng(0)

print(f'{"photons collected":>20s} {"fractional precision":>22s} {"= in ppm":>12s}')
for n_photons in (1e4, 1e6, 1e8, 1e10):
    precision = 1 / np.sqrt(n_photons)
    print(f'{n_photons:20,.0f} {precision:22.2e} {precision * 1e6:12,.0f}')

print()
print('An 8,400 ppm transit (Kepler-8 b) needs ~1.4e4 photons per measurement to see at 1-sigma.')
print('An 84 ppm transit (an Earth) needs ~1.4e8 -- ten thousand times more.')
print('THAT is why small planets are hard: the difficulty scales as depth^-2.')

In [ ]:
# Demonstrate it: simulate counting photons, and watch precision improve as 1/sqrt(N).
counts = np.array([1e2, 1e3, 1e4, 1e5, 1e6, 1e7])
measured = [np.std(rng.poisson(c, size=4000) / c) for c in counts]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.loglog(counts, measured, 'o', ms=8, color=plotting.SERIES[0], label='simulated Poisson draws')
ax.loglog(counts, 1 / np.sqrt(counts), '-', color=plotting.SERIES[1], label=r'theory  $1/\sqrt{N}$')
ax.set_xlabel('Photons collected per measurement')
ax.set_ylabel('Fractional scatter')
ax.set_title('Photon noise: the precision floor nothing can beat')
ax.legend()
plt.show()

print('The simulation sits exactly on the theory line. This is the budget you work within:')
print('everything in the light curves you analyse is a fight to stay near this floor.')

## 6. The facilities worth knowing

| Mission | Band | What it does | Why you care |
|---|---|---|---|
| **Kepler / K2** (2009–18) | Optical | Stared at one field for 4 years | Most of your data |
| **TESS** (2018–) | Optical | Nearly all-sky, 27-day sectors | Where new candidates are |
| **Gaia** (2013–25) | Optical | Astrometry for ~2 billion stars | Distances and stellar properties for *any* target |
| **Hubble** (1990–) | UV/opt/near-IR | High resolution above the atmosphere | Atmospheres, imaging |
| **JWST** (2021–) | Infrared | 6.5 m, cold, in deep space | Exoplanet atmospheres, early galaxies |
| **Chandra / XMM** | X-ray | Hot gas, accretion | Compact objects |
| **ALMA / VLA** | Radio/mm | Cold gas and dust | Planet formation |
| **Rubin (LSST)** (~2025–) | Optical | Whole southern sky every few nights | Time domain at enormous scale |
| **Roman** (~2027) | Infrared | Wide-field survey | Microlensing planets by the thousand |
| **LIGO/Virgo** | Gravitational waves | Not light at all | Merging compact objects |

Two of these are about to change what a hobbyist can do. **Rubin** will produce an alert
stream of ~10 million transient events *per night* — far more than professionals can inspect.
**Roman** will do microlensing at scale. Both mean vastly more public data than there are
people to look at it.

**Next:** `p4_the_data.ipynb` — what these instruments actually hand you.